In [1]:
!nvidia-smi

Tue May 12 08:22:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /kaggle/working

# PyTorch 2.2.0 + CUDA 12.1 (versione richiesta da icg_net)
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

# Sistema + numpy/scipy compatibili
!apt-get install -y libopenblas-dev -q
!pip install "numpy<2" "scipy==1.13.1" -q

print(" PyTorch e dipendenze base installate")
import torch
print(f"   torch: {torch.__version__}, CUDA disponibile: {torch.cuda.is_available()}")


/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 2.8 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.8 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 95.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 34.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 14.9 MB/s eta 0:00:0000:0100:01


In [3]:
%cd /kaggle/working

!rm -rf MinkowskiEngine
!git clone https://github.com/renezurbruegg/MinkowskiEngine.git
%cd MinkowskiEngine

import os, pathlib, glob

# === PATCH 1: ranges.hpp → no-op (ROOT CAUSE del conflitto nvtx3) ===
# ranges.hpp include il nvtx3.hpp bundled (vecchio) che entra in conflitto
# con nvtx3.hpp di CUDA 12.8 già incluso via cub/thrust.
# CUDF_FUNC_RANGE() è solo profiling NVTX → possiamo renderlo no-op.
pathlib.Path("src/3rdparty/cudf/detail/nvtx/ranges.hpp").write_text(
"""#pragma once
// Patched: disable cudf NVTX to fix conflict with CUDA 12.8+ bundled nvtx3.hpp
#ifndef CUDF_FUNC_RANGE
#define CUDF_FUNC_RANGE()
#endif
#ifndef CUDF_RANGE_PUSH
#define CUDF_RANGE_PUSH(a)
#endif
#ifndef CUDF_RANGE_POP
#define CUDF_RANGE_POP()
#endif
"""
)
print("✅ PATCH 1: ranges.hpp → no-op (nvtx3 conflict eliminato)")

# === PATCH 2: numpy.distutils rimosso in Python 3.12 ===
setup_path = pathlib.Path("setup.py")
text = setup_path.read_text()
if "numpy.distutils" in text:
    text = text.replace(
        "import numpy.distutils.system_info as sysinfo",
        "pass  # removed: numpy.distutils not available in Python 3.12"
    )
    dummy = """
class sysinfo:
    @staticmethod
    def get_info(name):
        return {"libraries": ["openblas"],
                "library_dirs": ["/usr/lib/x86_64-linux-gnu"],
                "include_dirs": ["/usr/include"]}

"""
    text = dummy + text
    setup_path.write_text(text)
    print("✅ PATCH 2: numpy.distutils shim aggiunto")

# === Compilazione con log completo ===
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['TORCH_CUDA_ARCH_LIST'] = '6.0+PTX;7.0;7.5;8.0;8.6'

print("\n🔨 Compilazione in corso (~5 min)... log in /tmp/mink_build.log")
!python setup.py bdist_wheel --dist-dir /kaggle/working/wheels \
    2>&1 | tee /tmp/mink_build.log | grep -E "(error:|Error:|FAILED|Successfully built)" | head -40

# Se fallisce, mostra i veri errori nvcc
wheels = glob.glob("/kaggle/working/wheels/*.whl")
if wheels:
    print(f"\n✅ Wheel creato: {wheels[0]}")
    !pip install {wheels[0]} -q
    print("✅ MinkowskiEngine installato!")
else:
    print("\n❌ Fallito. Errori nvcc reali:")
    !grep -E "^.*(error:).*" /tmp/mink_build.log | grep -v "^In file" | head -30
    print("\n--- Ultimi 30 righe del log ---")
    !tail -30 /tmp/mink_build.log


/kaggle/working
Cloning into 'MinkowskiEngine'...
remote: Enumerating objects: 5670, done.
remote: Counting objects: 100% (1745/1745), done.
remote: Compressing objects: 100% (483/483), done.
remote: Total 5670 (delta 1317), reused 1262 (delta 1262), pack-reused 3925 (from 2)
Receiving objects: 100% (5670/5670), 7.36 MiB | 38.86 MiB/s, done.
Resolving deltas: 100% (4449/4449), done.
/kaggle/working/MinkowskiEngine
✅ PATCH 1: ranges.hpp → no-op (nvtx3 conflict eliminato)
✅ PATCH 2: numpy.distutils shim aggiunto

🔨 Compilazione in corso (~5 min)... log in /tmp/mink_build.log

✅ Wheel creato: /kaggle/working/wheels/MinkowskiEngine-0.5.4-cp312-cp312-linux_x86_64.whl
✅ MinkowskiEngine installato!


In [4]:
%cd /kaggle/working

# Repo principale e benchmark
!git clone https://github.com/renezurbruegg/icg_net.git
!git clone https://github.com/renezurbruegg/icg_benchmark.git

# Dipendenze PyG (esatte per torch 2.2.0+cu121)
!pip install torch_geometric -q
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.2.0+cu121.html -q

# Dipendenze generali
!pip install hydra-core open3d trimesh networkx einops -q

print("✅ Dipendenze Python installate")


/kaggle/working
Cloning into 'icg_net'...
remote: Enumerating objects: 292, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 292 (delta 39), reused 281 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (292/292), 3.80 MiB | 29.44 MiB/s, done.
Resolving deltas: 100% (39/39), done.
Cloning into 'icg_benchmark'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 237 (delta 72), reused 229 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 794.91 KiB | 8.37 MiB/s, done.
Resolving deltas: 100% (72/72), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# Hydra >= 1.2 ha spostato compose/initialize dal modulo experimental
import subprocess, pathlib

icgnet_py = pathlib.Path("/kaggle/working/icg_net/icg_net/icg_net.py")
text = icgnet_py.read_text()

text = text.replace(
    "from hydra.experimental import compose, initialize",
    "from hydra import compose, initialize"
)
text = text.replace(
    "with initialize():",
    "with initialize(version_base=None):"
)
text = text.replace(
    "with initialize(config_path",
    "with initialize(version_base=None, config_path"
)

icgnet_py.write_text(text)
print("✅ Patch Hydra applicata")


✅ Patch Hydra applicata


In [6]:
%cd /kaggle/working/icg_net/icg_net/third_party/pointnet2
!pip install . -q
print("✅ PointNet2 installato")


/kaggle/working/icg_net/icg_net/third_party/pointnet2
  Preparing metadata (setup.py) ... done
✅ PointNet2 installato


In [7]:
%cd /kaggle/working/icg_net/icg_net/utils/mcubes

# mcubes usa Cython ma il file .pyx potrebbe non compilare in Python 3.12
# Proviamo prima la via Cython, con fallback al .cpp pre-generato
import os, subprocess, pathlib

setup_text = pathlib.Path("setup.py").read_text()

# Controlla se c'è già il .cpp pre-generato
cpp_exists = pathlib.Path("src/mcubes.cpp").exists()

if cpp_exists:
    # Usa il .cpp direttamente, senza Cython
    setup_text = setup_text.replace('"src/mcubes.pyx"', '"src/mcubes.cpp"')
    setup_text = setup_text.replace("cythonize(ext_modules)", "ext_modules")
    setup_text = setup_text.replace("from Cython.Build import cythonize\n", "")
    pathlib.Path("setup.py").write_text(setup_text)
    print("✅ Patch mcubes: uso del .cpp pre-compilato")
else:
    # Installa Cython e compila il .pyx
    subprocess.run(["pip", "install", "cython", "-q"], check=True)
    print("✅ Cython installato per compilare mcubes.pyx")

!python setup.py build_ext --inplace 2>&1 | tail -10
print("✅ mcubes compilato")


/kaggle/working/icg_net/icg_net/utils/mcubes
✅ Patch mcubes: uso del .cpp pre-compilato
In file included from /usr/local/lib/python3.12/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.12/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.12/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from /kaggle/working/icg_net/icg_net/utils/mcubes/src/mcubes.cpp:1198:
/usr/local/lib/python3.12/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -s

In [16]:
import sys, subprocess

# MinkowskiEngine è già installato come .whl compilato → possiamo riportare numpy a 2.x
# icg_net usa numpy.char e numpy.strings che esistono SOLO in numpy >= 2.0
print("🔄 Aggiorno numpy a 2.x (icg_net lo richiede)...")
!pip install "numpy==2.0.2" -q
!pip install loguru -q
print("✅ numpy aggiornato")

# Verifica che MinkowskiEngine regga ancora
try:
    import MinkowskiEngine as ME
    print(f"✅ MinkowskiEngine ancora funzionante: {ME.__version__}")
except ImportError as e:
    print(f"⚠️  MinkowskiEngine non importabile: {e}")
    print("   → Reinstalla il .whl se necessario")

# icg_net: aggiungi al path (evita pyproject.toml)
ICG_NET_PATH = '/kaggle/working/icg_net'
if ICG_NET_PATH not in sys.path:
    sys.path.insert(0, ICG_NET_PATH)

try:
    import icg_net
    print(f"✅ icg_net importato: {icg_net.__file__}")
except ImportError as e:
    print(f"❌ icg_net: {e}")

# icg_benchmark
result = subprocess.run(
    ["pip", "install", "-e", "/kaggle/working/icg_benchmark", "-q"],
    capture_output=True, text=True
)
print("✅ icg_benchmark OK" if result.returncode == 0 else f"❌ {result.stderr[-300:]}")


🔄 Aggiorno numpy a 2.x (icg_net lo richiede)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00
✅ numpy aggiornato
✅ MinkowskiEngine ancora funzionante: 0.5.4
✅ icg_net importato: /kaggle/working/icg_net/icg_net/__init__.py
✅ icg_benchmark OK


In [17]:
%cd /kaggle/working/icg_benchmark
!python scripts/download_data.py

# Trova il config.yaml reale
import subprocess, pathlib

result = subprocess.run(
    ['find', '/kaggle/working/icg_benchmark', '-name', 'config.yaml'],
    capture_output=True, text=True
)
print("Config trovati:\n", result.stdout)

configs = [p for p in result.stdout.strip().split('\n') if p]
if configs:
    CONFIG_PATH = configs[0]
    print(f"✅ Usando: {CONFIG_PATH}")
else:
    # Fallback: cerca anche in icg_net
    result2 = subprocess.run(
        ['find', '/kaggle/working/icg_net', '-name', 'config.yaml'],
        capture_output=True, text=True
    )
    print("Config in icg_net:\n", result2.stdout)
    configs = [p for p in result2.stdout.strip().split('\n') if p]
    CONFIG_PATH = configs[0] if configs else None
    print(f"CONFIG_PATH = {CONFIG_PATH}")


/kaggle/working/icg_benchmark
Downloading...
From: https://drive.google.com/uc?id=1w5TkzRt-aQwLud1CKozbw6TBuwExVf50
To: /kaggle/working/icg_benchmark/tmp.zip
100%|██████████████████████████████████████| 12.5M/12.5M [00:00<00:00, 33.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1RzdhOBX5GV42qbvmjuL31Nf1N90gT-ra
From (redirected): https://drive.google.com/uc?id=1RzdhOBX5GV42qbvmjuL31Nf1N90gT-ra&confirm=t&uuid=7280b782-fe2b-4b1c-8b4b-63c1666eb900
To: /kaggle/working/icg_benchmark/tmp.zip
100%|███████████████████████████████████████| 36.2M/36.2M [00:00<00:00, 178MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hmL5zeByQuWlytW1ZDDLyHy4pLjN-3VN
To: /kaggle/working/icg_benchmark/tmp.zip
100%|██████████████████████████████████████| 18.0M/18.0M [00:00<00:00, 34.0MB/s]
Done downloading data.
Config trovati:
 /kaggle/working/icg_benchmark/data/icgnet/51--0.656/config.yaml

✅ Usando: /kaggle/working/icg_benchmark/data/icgnet/51--0.656/config.yaml


In [19]:
import pathlib, os
icgnet_py = pathlib.Path("/kaggle/working/icg_net/icg_net/icg_net.py")
text = icgnet_py.read_text()
# Patch: sostituisce initialize() senza config_path con initialize_config_dir()
# che accetta un percorso assoluto alla directory del config
old_block = '''        if isinstance(config, str):
            with initialize(version_base=None):
                cfg = compose(config_name=config, return_hydra_config=True, overrides=[])'''
new_block = '''        if isinstance(config, str):
            import os as _os
            from hydra import initialize_config_dir
            _config_dir = _os.path.dirname(_os.path.abspath(config))
            _config_name = _os.path.splitext(_os.path.basename(config))[0]
            with initialize_config_dir(version_base=None, config_dir=_config_dir):
                cfg = compose(config_name=_config_name, return_hydra_config=True, overrides=[])'''
if old_block in text:
    text = text.replace(old_block, new_block)
    icgnet_py.write_text(text)
    print("✅ Patch Hydra initialize_config_dir applicata")
else:
    print("⚠️  Blocco non trovato — mostra il contenuto attuale:")
    # Cerca le righe con initialize
    for i, line in enumerate(text.splitlines()):
        if 'initialize' in line or 'compose' in line:
            print(f"  {i+1}: {line}")

✅ Patch Hydra initialize_config_dir applicata


In [21]:
import pathlib, subprocess

CONFIG_DIR = "/kaggle/working/icg_benchmark/data/icgnet/51--0.656"

# 1. Trova il checkpoint reale scaricato
result = subprocess.run(
    ['find', '/kaggle/working/icg_benchmark', '-name', '*.ckpt'],
    capture_output=True, text=True
)
print("Checkpoint trovati:", result.stdout)
CKPT_PATH = result.stdout.strip().split('\n')[0]
print(f"Uso: {CKPT_PATH}")

# 2. Patcha il config.yaml sostituendo il path hardcoded di Rene
config_yaml = pathlib.Path(f"{CONFIG_DIR}/config.yaml")
text = config_yaml.read_text()

# Sostituisce qualsiasi path che finisce in checkpoint.ckpt
import re
text_patched = re.sub(
    r'checkpoint:\s*.*checkpoint\.ckpt',
    f'checkpoint: {CKPT_PATH}',
    text
)

if text_patched != text:
    config_yaml.write_text(text_patched)
    print(f"✅ config.yaml patchato con: {CKPT_PATH}")
else:
    # Mostra le righe con 'checkpoint' per debug
    print("⚠️  Pattern non trovato. Righe con 'checkpoint' nel yaml:")
    for line in text.splitlines():
        if 'checkpoint' in line.lower():
            print(f"  {line}")


Checkpoint trovati: /kaggle/working/icg_benchmark/data/icgnet/51--0.656/checkpoint.ckpt

Uso: /kaggle/working/icg_benchmark/data/icgnet/51--0.656/checkpoint.ckpt
✅ config.yaml patchato con: /kaggle/working/icg_benchmark/data/icgnet/51--0.656/checkpoint.ckpt


In [22]:
import sys, torch, numpy as np

# Pulisci cache moduli (necessario dopo la patch di icg_net.py)
for mod in list(sys.modules.keys()):
    if 'icg_net' in mod:
        del sys.modules[mod]

# Re-import
if '/kaggle/working/icg_net' not in sys.path:
    sys.path.insert(0, '/kaggle/working/icg_net')

from icg_net import ICGNetModule, get_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model: ICGNetModule = get_model(CONFIG_PATH, device=device)
print("✅ Modello caricato!")

# Smoke test con pointcloud sintetica
N = 500
pts = torch.randn(N, 3).float() * 0.1
normals = pts / pts.norm(dim=1, keepdim=True)

out = model(
    pts, normals=normals,
    grasp_pts=pts, grasp_normals=normals,
    n_grasps=32, each_object=True,
    return_meshes=False, return_scene_grasps=True,
)
print(f"✅ Inferenza completata: {type(out)}")
print(out)


mcubes not installed... Mesh extraction will not work.
Device: cuda
Created CombinedFeatureVolume
Sparse Backbone:  <class 'icg_net.minkowski.res16unet.Res16UNetTiny'>
Dense Backbone:  <class 'icg_net.model.feature_volume.dense.DenseFeatureExtractor'>
Dense Interpolator:  <class 'icg_net.model.interpolation.interpolator.DenseVolumeInterpolator'>
Dense feature shapes:  [128, 128, 128, 64, 48]
FOUND CHECKPOINT. Loading from  /kaggle/working/icg_benchmark/data/icgnet/51--0.656/checkpoint.ckpt


2026-05-12 09:28:23.704 | WARNING  | icg_net.utils.checkpoint:load_checkpoint_with_missing_or_exsessive_keys:51 - Key not found, it will be initialized randomly: criterion.empty_weight
2026-05-12 09:28:23.713 | WARNING  | icg_net.utils.checkpoint:load_checkpoint_with_missing_or_exsessive_keys:73 - excessive key: criterion.empty_weight


✅ Modello caricato!
✅ Inferenza completata: <class 'icg_net.typing.types.ModelPredOut'>
ModelPredOut(class_predictions=tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
 

/kaggle/working/icg_net/icg_net/utils/grasps.py:215: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at ../aten/src/ATen/native/Cross.cpp:63.)
  x_axis = torch.cross(gravity, y_axis)# torch.stack([-y_axis[:,1], y_axis[:,0], 0*y_axis[:,0]], dim=-1)
